In [ ]:
# Resolve paths relative to the repository root so this notebook runs from the repo
# root, from notebooks/, or on Colab after cloning the repo:
#   !git clone https://github.com/yasmeenhany/media-bias-detection.git
#   %cd media-bias-detection
import os
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
os.chdir(REPO_ROOT)
print("Working directory:", REPO_ROOT)

In [1]:
import pickle

In [18]:
import pandas as pd

PATH_sg1 = "./data/final_labels_SG1.xlsx"
PATH_sg2 = "./data/final_labels_SG2.xlsx"
df_sg1 = pd.read_excel(PATH_sg1)
df_sg2 = pd.read_excel(PATH_sg2)
df_sg1.rename(columns={'text': 'sentence', 'label_bias': 'Label_bias'}, inplace=True)
df_sg2.rename(columns={'text': 'sentence', 'label_bias': 'Label_bias'}, inplace=True)
df_sg1.head()

,sentence,news_link,outlet,topic,type,Label_bias,label_opinion,biased_words
0,The Republican president assumed he was helpin...,http://www.msnbc.com/rachel-maddow-show/auto-i...,msnbc,environment,left,Biased,Expresses writer’s opinion,[]
1,Though the indictment of a woman for her own p...,https://eu.usatoday.com/story/news/nation/2019...,usa-today,abortion,center,Non-biased,Somewhat factual but also opinionated,[]
2,Ingraham began the exchange by noting American...,https://www.breitbart.com/economy/2020/01/12/d...,breitbart,immigration,right,No agreement,No agreement,['flood']
3,The tragedy of America’s 18 years in Afghanist...,http://feedproxy.google.com/~r/breitbart/~3/ER...,breitbart,international-politics-and-world-news,right,Biased,Somewhat factual but also opinionated,"['tragedy', 'stubborn']"
4,The justices threw out a challenge from gun ri...,https://www.huffpost.com/entry/supreme-court-g...,msnbc,gun-control,left,Non-biased,Entirely factual,[]


In [3]:
with open("./results/ernie_None_results.pickle", "rb") as f:
    data = pickle.load(f)

In [8]:
for k in data.keys(): # Each key represents each fold
    print(data[k][0]) # Each list element is (idx, sentence, true_label, pred_label, pred_value)

(0, 'Though the indictment of a woman for her own pregnancy loss is unusual in Alabama, it is not unusual for prosecutors to charge people with murder even if they never killed anyone. ', 0, 0, array([ 1.1116599, -1.1331686], dtype=float32))
(0, 'The Republican president assumed he was helping the industry at the expense of the environment – a trade-off Trump was happy to make since he rejects climate science anyway. ', 1, 1, array([-0.1705498 ,  0.53706723], dtype=float32))
(0, 'After the airstrike that killed Iranian Gen. Qasem Soleimani, there were reports that U.S. Customs and Border Patrol detained dozens of people at the Blaine, Washington, port of entry over the weekend.', 0, 0, array([ 0.785249 , -1.8700479], dtype=float32))
(0, 'Investigators believe parents would use falsified records so their children could attend schools that would otherwise require all students to be vaccinated, Dart told WBBM-TV.', 0, 0, array([ 0.7360011, -1.8549842], dtype=float32))
(0, 'The justices th

In [11]:
biased_correct = []
unbiased_correct = []
biased_incorrect = []
unbiased_incorrect = []

for k in data.keys():

    for row in data[k]:

        sentence, true_label, pred_label, pred_value = row[1], row[2], row[3], row[4]

        if true_label == pred_label:

            if true_label == 1:
                biased_correct.append((sentence, abs(pred_value[0]-pred_value[1])))
            else:
                unbiased_correct.append((sentence, abs(pred_value[0]-pred_value[1])))

        else:
            if true_label == 1:
                biased_incorrect.append((sentence, abs(pred_value[0]-pred_value[1])))
            else:
                unbiased_incorrect.append((sentence, abs(pred_value[0]-pred_value[1])))


Get confidently correct/incorrect sentences

In [12]:
biased_correct = sorted(biased_correct, key=lambda x: x[1], reverse=True)
unbiased_correct = sorted(unbiased_correct, key=lambda x: x[1], reverse=True)
biased_incorrect = sorted(biased_incorrect, key=lambda x: x[1], reverse=True)
unbiased_incorrect = sorted(unbiased_incorrect, key=lambda x: x[1], reverse=True)

In [14]:
biased_correct[:5]

[('Nevertheless, the media jihad continues, and that includes NBC shamelessly running a debunked and deceptive smear story that had been reported on and debunked all the way back in May.',
  3.6959786),
 ('Trump’s profound megalomania, impulsiveness, and defective character is what got us where we are today. If we are to have any shot at a real recovery, he needs to go.',
  3.6761773),
 ('He’s a 78-year-old white male millionaire seeking the affections of the wokesters, a gruff socialist with bedraggled hair and\xa0terrifying policy proposals.',
  3.6722558),
 ("I'm also struck by Blackburn's style of needlessly toxic politics, once again questioning the patriotism of a decorated U.S. Army combat veteran who earned a Purple Heart.",
  3.6717463),
 ('In an era in which domestic terrorism is an increasingly deadly societal scourge, it’s dangerous when a sitting American president lies to rabid followers with tales of infants being executed in medical facilities.',
  3.6469917)]

In [15]:
unbiased_correct[:5]

[('The Georgian Orthodox Church Patriarchy said all Easter services would be held in a traditional manner this week but parishioners would be required to maintain social distancing between each other to stem transmission of the virus.',
  3.2466407),
 ('More than 400 women have applied to attend a candidate training session this weekend in Washington hosted by Emily’s List, an organization that works to elect women who support abortion rights, said spokeswoman Tonya Williams.',
  3.1710494),
 ('Democrats blocked a coronavirus package on Sunday that would provide economic relief to businesses and Americans suffering from the impact of the coronavirus outbreak',
  3.1492853),
 ('Sen. Bernie Sanders finished ahead of Joe Biden in the first major national poll of the year that did not find the former vice president leading the pack of potential 2020 Democratic presidential candidates. ',
  3.1367326),
 ('Sen. Steve Daines, a Republican from Montana, announced during the March for Life rall

In [16]:
biased_incorrect[:5]

[('The number of April background checks to purchase a firearm showed Americans flocking to gun stores in high numbers for the second month in a row amid worries about the coronavirus pandemic.',
  3.0985239),
 ('Four states will hold their primary contests on Tuesday night, despite Georgia and Louisiana each postponing their elections previously scheduled for a few weeks later over concerns of the spreading Wuhan virus.',
  2.9968643),
 ('Tens of millions of children under 12 months are potentially at risk for diseases such as diphtheria and polio as the Chinese coronavirus pandemic interrupts routine vaccinations, according to data published by global public health experts on Friday.',
  2.923818),
 ('Hispanic foreigners attending the town hall — Fox News Latino did not say whether they were legal or illegal aliens — told Obama they feared that federal Judge Andrew Hansen’s stay on Obama’s executive order will force them to return to their home countries.',
  2.8456645),
 ('The docto

In [17]:
unbiased_incorrect[:5]

[('Fox News has gone so deep into white nationalism that Donald Trump now believes it’s how he’ll win in 2020',
  2.7838821),
 ('While, in the wake of the El Paso murders, the media and most of our politicians are acting shocked that there are armed racists in America who are trying to terrify people of color, the reality is that it’s pretty much always been that way here in the United States.',
  2.7289917),
 ('The white nationalist who said “Hail Trump” and “hail our people” during a conference in Washington D.C. on Saturday — and who received straight-armed Nazi-like salutes in response ...',
  2.7242017),
 ('Yet the drumbeat for censorship goes on as the politicians whose ideas find favor with the mainstream media want to make sure theirs are the only ideas you ever get to hear. Today it will be about money, but only because money is the means to spread ideas.',
  2.6596613),
 ('Of course, social distancing will be inconceivable for football players and the results all too predicta

In [31]:
from collections import defaultdict

label_opinion_cnt = defaultdict(int)
for sentence, score in biased_correct:

    row = df_sg1[df_sg1["sentence"] == sentence]
    label_opinion = row["label_opinion"].values[0]
    label_opinion_cnt[label_opinion] += 1

print("Biased sentences which were predicted as biased")
for key, value in label_opinion_cnt.items():
    print(f"{key}: {value}")

print()


label_opinion_cnt = defaultdict(int)
for sentence, score in biased_incorrect:

    row = df_sg1[df_sg1["sentence"] == sentence]
    label_opinion = row["label_opinion"].values[0]
    label_opinion_cnt[label_opinion] += 1

print("Biased sentences which were predicted as unbiased")
for key, value in label_opinion_cnt.items():
    print(f"{key}: {value}")

print()

label_opinion_cnt = defaultdict(int)
for sentence, score in unbiased_correct:

    row = df_sg1[df_sg1["sentence"] == sentence]
    label_opinion = row["label_opinion"].values[0]
    label_opinion_cnt[label_opinion] += 1

print("Unbiased sentences which were predicted as unbiased")
for key, value in label_opinion_cnt.items():
    print(f"{key}: {value}")

print()

label_opinion_cnt = defaultdict(int)
for sentence, score in unbiased_incorrect:

    row = df_sg1[df_sg1["sentence"] == sentence]
    label_opinion = row["label_opinion"].values[0]
    label_opinion_cnt[label_opinion] += 1

print("Unbiased sentences which were predicted as biased")
for key, value in label_opinion_cnt.items():
    print(f"{key}: {value}")

print()

Biased sentences which were predicted as biased
Expresses writer’s opinion: 291
Somewhat factual but also opinionated: 195
No agreement: 43
Entirely factual: 9

Biased sentences which were predicted as unbiased
Somewhat factual but also opinionated: 118
No agreement: 33
Expresses writer’s opinion: 38
Entirely factual: 19

Unbiased sentences which were predicted as unbiased
Entirely factual: 505
No agreement: 64
Somewhat factual but also opinionated: 62
Expresses writer’s opinion: 30

Unbiased sentences which were predicted as biased
Expresses writer’s opinion: 34
Entirely factual: 59
Somewhat factual but also opinionated: 27
No agreement: 19



In [22]:
for i, row in df_sg1.iterrows():

    if row["sentence"] == biased_incorrect[1][0]:
        print(row)

sentence         Four states will hold their primary contests o...
news_link        https://thefederalist.com/2020/03/16/wuhan-vir...
outlet                                                  federalist
topic                                               elections-2020
type                                                         right
Label_bias                                                  Biased
label_opinion                Somewhat factual but also opinionated
biased_words                                    ['Wuhan', 'virus']
Name: 1516, dtype: object


In [23]:
for i, row in df_sg1.iterrows():

    if row["sentence"] == biased_incorrect[2][0]:
        print(row)

sentence         Tens of millions of children under 12 months a...
news_link        https://www.breitbart.com/health/2020/05/22/re...
outlet                                                   breitbart
topic                                                     vaccines
type                                                         right
Label_bias                                                  Biased
label_opinion                Somewhat factual but also opinionated
biased_words                            ['Chinese', 'coronavirus']
Name: 735, dtype: object
